In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3' 
import nugget
import time
import torch
from nugget.utils import jax_bridge
from nugget.samplers.sim_weights import event_arrays, generation_params, get_simulation_weights
from nugget.samplers.flux_weights import get_flux_weights
from nugget.utils.grad_accumulation import disjoint_batches

In [2]:
device = 'cuda:0' #specify device if you want to use GPU
jax_bridge.configure_jax(device=device) #configure jax (used by pyForwardFolding for the flux weights) before it is first imported

jax_bridge: JAX default device is cuda:0


<module 'jax' from '/u/kristiantcho/ptmp/nugget_env/lib/python3.12/site-packages/jax/__init__.py'>

In [3]:
# lightsabre = nugget.surrogates.LightSabre.LightSabre(
#                 device=device,
#                 use_poisson=False, 
#                 particle_mode='track',
#                 ) # Initialize the LightSabre surrogate (used by the trigger)
# signal_sampler = nugget.samplers.cyl_sampler.CylinderSampler(
#     device=device,
#     event_type='signal', 
#     domain_size=1200, 
#     E_min=1e2, 
#     E_max=1e6, 
#     energy_dist='log_uniform', 
#     find_exact_intersection=False,
#     random_position_along_ray=True,
#     uniform_zenith_sampling=True,
#     cylinder_radius=800,
#     cylinder_height=1000,
#     cylinder_center=[0, 0, 0],
# ) # Initialize the signal event sampler
signal_sampler = nugget.samplers.range_sampler.RangeSampler(
    device=device,
    E_min=1e2,
    E_max=1e6,
    energy_dist='log_uniform',
    injection_radius=1100,
    chunk_size=10000,
) # Initialize the signal event sampler

### Fixed event sample with precalculated weights

10000 events are sampled once. Their simulation weights (A_eff / (N_gen q(E) p(Omega)), normalised to all 10000 events) and flux weights (Phi and dPhi/dparam from friEnd + pyForwardFolding) do not depend on the geometry here, so they are computed once. Every optimization step then uses all 10000 events in 10 disjoint batches of 1000, with the gradient accumulated by `AnalysisAccumulator`: the histograms are pooled over the batches, the Fisher head is differentiated once on the pooled histograms, and that gradient is pushed back through each batch in turn.

In [4]:
num_events_total = int(1e5)
signal_events = signal_sampler.sample_events(num_events_total)

/ptmp/mpp/kristiantcho/nugget/nugget/samplers/range_sampler.py:52: UserWarning: torch.searchsorted(): boundary tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous boundary tensor if possible. This message will only appear once per program. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/BucketizationUtils.h:38.)
  idx = torch.searchsorted(self.r_out, r.contiguous())


In [5]:

# batch_size = 500
live_time = 10 * 365.25 * 24 * 3600  # 10 years in seconds
friend_config = '../other/friend_config.yaml'
pyff_config = '../other/pyff_config.yaml'

# pass filename=... to both if nugget/assets/data is not staged on this machine
xsec = nugget.losses.effective_area.CrossSection()
transmission = nugget.losses.effective_area.TransmissionProb()

time_start = time.time()
ev = event_arrays(signal_events)  # arrival convention, as the transmission and atmospheric fluxes expect

print(f'event_arrays took {time.time() - time_start:.2f} s')
time_start = time.time()
sim_weights = get_simulation_weights(
    ev['energy'], ev['cos_zenith'],
    # signal_sampler.cylinder.radius, signal_sampler.cylinder.height,
    xsec, transmission, batched=True, column_depth=ev['column_depth'],
    **generation_params(signal_sampler),
) # GeV cm^2 sr, normalised to N_gen = num_events_total
print(f'get_simulation_weights took {time.time() - time_start:.2f} s')
time_start = time.time()
flux, grad_flux = get_flux_weights(
    ev['energy'], ev['zenith'], ev['azimuth'],
    friend_config, pyff_config, device=device,
) # GeV^-1 cm^-2 s^-1 sr^-1, and its gradient w.r.t. each flux parameter
print(f'get_flux_weights took {time.time() - time_start:.2f} s')
print('flux parameters:', list(grad_flux))
print(f'expected events before trigger in {live_time / 3.15576e7:.0f} yr: {(live_time * sim_weights * flux).sum().item():.4g}')

event_arrays took 2.62 s
get_simulation_weights took 12.22 s


Running step: powerlaw_weight
  Config: {'norm': 1e-18, 'gamma': 2.0, 'e0': 100000.0, 'output_keys': {'weight': 'weight_powerlaw'}}
  Done.
Running step: mceq_spline_weight
  Config: {'spline_path': '/u/kristiantcho/ptmp/nugget/nugget/assets/data/LAT47-LON-127_coszsteps256-azimuthsteps256_SIBYLL-2.3c-GSF.pickle', 'flux_types': ['conv', 'pr'], 'output_keys': {'conv': 'mceq_conv_SIBYLL23c_GSF', 'pr': 'mceq_pr_SIBYLL23c_GSF'}}
  Done.
Running step: mceq_spline_weight
  Config: {'spline_path': '/u/kristiantcho/ptmp/nugget/nugget/assets/data/LAT47-LON-127_coszsteps256-azimuthsteps256_SIBYLL-2.3c-H4a.pickle', 'flux_types': ['conv', 'pr'], 'output_keys': {'conv': 'mceq_conv_SIBYLL23c_H4a', 'pr': 'mceq_pr_SIBYLL23c_H4a'}}
  Done.
get_flux_weights took 20.44 s
flux parameters: ['astro_index', 'astro_norm', 'conv_norm', 'delta_gamma', 'lambda_int', 'prompt_norm']
expected events before trigger in 10 yr: 6.802e+07


In [6]:
geometry = nugget.geometries.DynamicString.DynamicString(
                device=device,
                hex_type='hexagonal',
                domain_size=500,  # Size of detector domain
                dim=3,  # 3D geometry
                n_strings=70,  # Initial number of detector strings
                points_per_string=20,  # Number of PMTs/sensors per string
                custom_z_spacing=40,
            ) # Initialize the dynamic string geometry object

visualizer = nugget.utils.vis_tools.Visualizer(
    device=device,
    dim=3, 
    domain_size=1400,
    gif_temp_dir='./gif_temp_analysis',  # Directory for storing animation frames
    ) # Initialize the visualizer for live visualization of the geometry and optimization process

In [7]:
# using here the hybrid optimizer which handles the gradient accumulation over event batches
optimizer = nugget.utils.hybrid_optimizer.HybridOptimizer(
    device=geometry.device, 
    geometry=geometry,
    visualizer=visualizer,
    use_alm=True,  # Use Augmented Lagrangian Method for constraints
    sigmoid_losses=True,
)
# initialize optimizer and create the initial geometry, specifying which parameters to optimize and their learning rates
optimizer.init_geometry(
    opt_list=[('string_xy', 5)],  # Learning rates for string xy positions
)

Optimizing string_xy with torch.Size([70, 2]) shape


In [8]:
# initialize loss functions 
HIT_CKPT   = './flow_models/best_mc_hit_model_v1r.pt'
LY_CKPT    = './flow_models/best_mc_ly_muon_flow_model_v1r.pt'
ATIME_CKPT = './flow_models/best_mc_atime_muon_flow_model_v1r.pt'
nusmoothie = nugget.surrogates.NuSmoothie.NuSmoothie(
    device=device,
    domain_size=10000,          # overwritten per model by load_model
    hit_checkpoint=HIT_CKPT,
    ly_checkpoint=LY_CKPT,
    atime_checkpoint=ATIME_CKPT,
    n_steps=8,
    ly_n_samples=1,
    batch_size=100000,
    grad_batch_size=2000000,
    # ly_prune_below =1e-2
)


# Local String Repulsion: Prevents strings from getting too close to each other
local_string_repulsion_penalty = nugget.losses.geometry_penalties.LocalStringRepulsionPenalty(
    device=geometry.device
)

# ROV Penalty: Accounts for physical constraints from Remotely Operated Vehicle
rov_penalty = nugget.losses.geometry_penalties.ROVPenalty(
    device=geometry.device, 
    rov_rec_width=230,  # ROV dimensions
    rov_height=160, 
    rov_tri_length=160
)

# === ANALYSIS LOSS ===
# Energy resolution from the per-OM Fisher network. geometry_grads='direct' keeps the graph from the
# string positions to each event's resolution, which the analysis loss bins with; 'chunked' only
# attaches the position gradient to the network loss's own scalar, not to the per-event resolutions.
om_energy_resolution_loss = nugget.losses.fisher_info_om_net.OMFisherNetLoss(
    device=geometry.device,
    resolution_type='energy',
    geometry_grads='direct',
).load_model('./fisher_models/best_om_muon_fisher_net_v3mc.pt')

# Detection efficiency: smooth sliding-bar trigger on the LightSabre light yield
# trigger_loss = nugget.losses.trigger.TriggerLoss(
#     device=geometry.device,
#     light_yield_threshold=6.0,
#     distance_bar_length=550.0,
#     distance_bar_step=50.0,
#     min_points_threshold=30.0,
#     use_hard_cuts=False,  # hard cuts would block the gradient
# )
trigger_loss = nugget.losses.photon_trigger.PhotonCountTrigger(
    device=geometry.device,
    n_photons=10,
    softness=0.2,
    cut='sigmoid',
    # photons='expected',
    count='photons',
    ly_prune_below=1e-3
)

# Forward-folding analysis: Fisher information of the flux parameters from the binned, weighted events
analysis_loss = nugget.losses.analysis_loss.AnalysisLoss(
    device=geometry.device,
    trigger_loss=trigger_loss,
    resolution_loss=om_energy_resolution_loss,
    xsec=xsec,
    transmission=transmission,
)

HitClassifier: context_dim=11, width=256, depth=8, params=1,060,609
FlowMatchLY: context_dim=11, width=256, depth=8, params=2,441,729
load_model: particle_mode not found in checkpoint; using default
FlowMatchLY: context_dim=11, width=256, depth=12, params=3,627,521
NeuralSabre: particle_mode=track, patd_mode=False, pmt_mode=sum, n=1.33
  hit    context_dim= 11 device=cuda:0 dtype=torch.float64
  ly     context_dim= 11 device=cuda:0 dtype=torch.float64
  atime  context_dim= 11 device=cuda:0 dtype=torch.float64
  OM template: 16 PMT directions from /ptmp/mpp/kristiantcho/nugget/nugget/other/800_40_40_geom.csv
  all 3 models share one context (11 features) -- built once per call
OMFisherNetLoss: params ['energy', 'zenith', 'azimuth'], target_dim=6, width=256, depth=10, weights=1,325,830


In [9]:
# set up loss function dictionary, parameters, weights, and sigmoid list for optimization
batch_size=500
loss_func_dict = {
    'fisher_analysis_loss': analysis_loss,                            # torch, accumulated over batches
    'local_string_repulsion_penalty': local_string_repulsion_penalty, # torch
    'rov_penalty': rov_penalty,                                       # torch
}

# Repeat k of every step gets batch k of a fresh permutation of the 10000 events (disjoint within a step).
# The sim weights are normalised to all 10000 events, so each batch scales them up by 10000 / 1000.
batch_hook = disjoint_batches(
    {
        'signal_event_params': signal_events,
        'analysis_sim_weights': sim_weights,
        'analysis_flux_weights': flux,
        'analysis_grad_flux_weights': grad_flux,
    },
    batch_size=batch_size,
    # rescale_keys=('analysis_sim_weights',),
)

loss_params = {
    # analysis loss
    'signal_sampler': signal_sampler,  # only read for the generation settings (energy range for the bins)
    'num_events': batch_size,
    'live_time': live_time,
    'friend_config': friend_config,
    'pyff_config': pyff_config,
    'analysis_binning_var_name': ['energy', 'zenith'],  # bins in log10(E)
    'analysis_num_bins': 30,
    'analysis_signal_flux_var_names': ['astro_norm'],
    'analysis_optimality': 'a',
    'use_relative_energy': False,  # the analysis loss needs the absolute energy resolution
    'fisher_info_chunk_size': 100000,  # (event, OM) rows per Fisher-net pass
    'analysis_energy_res_selection': True,
    # 'analysis_energy_res_threshold': 1.0,   # sigma_E / E where the weight is 0.5 (default)
    # 'analysis_energy_res_softness': 0.1,    # width of the cut, in ln(sigma_E / E) (default)

    # trigger
    'signal_surrogate_func': nusmoothie.light_yield_surrogate,
    'use_batched_surrogate': True,
    'use_batched_trigger': True,
    'binned_trigger_batch_size': 10000,  # events per trigger chunk, lower if it runs out of memory
    'analysis_perfect_efficiency': True,  # if True, skips the trigger and assumes every event is detected

    # penalties
    'max_radius': 80,  # Maximum radius for string repulsion penalty
    'num_angles': 360,  # Number of angles (divided into 360 degrees) to test for rov
    'rov_alt_mode': True,  # Whether to use alternative mode for rov penalty (see rov_penalty.py for details)
    'local_sharpness': 10,  # Sharpness parameter for local string repulsion
    'rov_soft_inside':True, # ROV penalty parameters
    'rov_inside_sharpness': 5,
    'rov_angle_softmin_tau': 0.05,
    'rov_angle_chunk_size': 360,
    'rov_inside_use_softplus': True,
    'constraints_list': [ 
                'rov_penalty', 
                'local_string_repulsion_penalty', 
                ],  # Constraints to enforce with ALM

    # gradient accumulation: 10 disjoint batches of 1000 events per step
    'loss_repeats':      {'fisher_analysis_loss': 10000//batch_size},  # number of disjoint batches to accumulate for each step
    'loss_accumulators': {'fisher_analysis_loss': 'analysis'},
    'on_repeat':         {'fisher_analysis_loss': batch_hook},
    'loss_reduction':    {'fisher_analysis_loss': 'mean'},  # batches added up gives the full histogram
    # 'loss_devices':    {'fisher_analysis_loss': 'cuda:2'},
    'empty_cache_per_repeat': True,
}   

loss_weights_dict = {
    'fisher_analysis_loss': 0.1,  # tune so that softness * weight * loss stays roughly within [-5, 5]
    'local_string_repulsion_penalty': 0.1,
    'rov_penalty': 1,
} # weights for each loss function, used in the optimization process (before applying ALM and sigmoid)

# List of loss functions that should use sigmoid activation before being applied, helps with loss balancing and scaling
loss_sigmoid_list = [
    'fisher_analysis_loss',
    'local_string_repulsion_penalty',
    'rov_penalty',
]

In [10]:
# specify which plots to generate live and their settings

plot_types = [
    'loss_components',           # Loss function values over time
    'flux_variance_history',     # Error on the signal flux parameters (the analysis loss) over time
    'string_xy_rov_penalty',
    'string_xy_local_string_repulsion_penalty',
    'string_history',
    # 'alm_lambda',
    'energy_resolution_hist',
]

vis_kwargs = {
    'plot_types': plot_types,
    'gif_plot_selection': plot_types,  # Which plots to include in animation
    'rov_penalty_func': rov_penalty,
    'plot_geom_contour_only': True,
    'rov_draw_safe_space_union': True,
    'rov_union_per_space_colors':True,
    'rov_draw_safe_space_on_violations':False,
    'flux_variance_as_sigma': True,  # plot the sum of 1-sigma errors rather than its square
    'energy_resolution_hist_bins': 20,
}

In [ ]:
# Begin optimization process, combining all the configurations

optimizer.visualizer.cleanup_gif_temp_files()
geom_dict = optimizer.optimize(
    clear_cuda_cache=True,                 # Clear torch GPU memory cache between iterations
    loss_func_dict=loss_func_dict,          # Dictionary of loss functions to use
    loss_weights_dict=loss_weights_dict,    # Weights for combining multiple losses
    loss_params_dict=loss_params,           # Parameters for loss function computation
    n_iter=2000,                           # Maximum number of optimization iterations
    vis_kwargs=vis_kwargs,                 # Visualization parameters
    print_freq=5,                          # Print progress every N iterations
    vis_freq=10,                           # Create plots every N iterations
    gif_freq=5,                           # Save animation frames every N iterations
    sigmoid_loss_list=loss_sigmoid_list,         # Which losses to apply sigmoid to (for better optimization dynamics)
    revert_on_nan=True,                      # Revert geometry if loss becomes NaN
    max_nan_retries=10,                          # Maximum number of consecutive NaN reverts before stopping
    # save_geom_folder='./analysis_geoms',  # Directory to save geometry checkpoints
    # save_geom_freq=100,                      # Save geometry every N iterations
    save_best_geom_file = './best_analysis_geom_2.pkl',  # File to save best geometry found
)

GIF temporary files cleanup completed.
HybridOptimizer: gradient accumulation plan
  fisher_analysis_loss: AnalysisAccumulator, 20x on cuda:0 (primary) [mean]


In [ ]:
# empty the GPU cache after optimization to free up memory
import gc
gc.collect()
torch.cuda.empty_cache()